# Preprocessing

### Load and Convert Fine-tuned Transformer Model to TransformerLens

In [ ]:
from src import load_finetuned_model

base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")

model.eval()

## Find the data with the correct answer

In [ ]:
from src import filter_correct_data
import pandas as pd

dataset_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_corrected.csv"
filtered_data_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered_AOS.csv"
test_df = pd.read_csv(dataset_path)

df_filtered = filter_correct_data(model, test_df, 
                                  "original_sentence", "original_triplet", filter_mode="AOS", filter_only_correct=False, 
                                  save_path=filtered_data_path)

## Create EAP Dataset

#### Building the Dataset

In [ ]:
import src
import importlib
importlib.reload(src.utils)

from src.utils import build_eap_dataset
import pandas as pd


filtered_data = pd.read_csv(
    "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered_S.csv")

eap_df = build_eap_dataset(
    model=model,
    df=filtered_data,
    sentence_col="original_sentence",
    triplet_col="original_triplet",
    corrupted_col="counterfact3_modified",
    corrupted_triplet_col="counterfact_triplet3_modified",
    suffix="[S]",
    filer_same_length_counterfactuals=True
)
eap_df.to_csv("eap_dataset/eap_dataset_sentiment_multitokens.csv", index=False)

# EAP-IG

In [ ]:
import gc
import torch

gc.collect()

torch.mps.empty_cache()

In [ ]:
import argparse
import ast
import os
from functools import partial
from random import random
from typing import Optional

import pandas as pd
import transformers
from torch.utils.data import Dataset, DataLoader
import torch
from typing_extensions import Tuple, List, Union

from eap.graph import Graph
from eap.evaluate import evaluate_graph, evaluate_baseline, evaluate_baseline_multitoken, evaluate_graph_multitoken
from eap.attribute import attribute
from src.utils import build_eap_dataset
from src import load_finetuned_model, filter_correct_data
from src.metric import logit_diff

In [ ]:
# load model
base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True
model.cfg.ungroup_grouped_query_attention = True

In [ ]:
# load dataset
ds = pd.read_csv("eap_dataset/eap_dataset_aspect_multitokens.csv")

In [ ]:
g = Graph.from_model(model)

In [ ]:
baseline = evaluate_baseline_multitoken(
    model,
    df=ds,
    metrics=[logit_diff],
    run_corrupted=False,
    batch_size=4
)
print(f"Original performance is logit_dif={baseline}")

In [ ]:
attribute(
    model=model,
    graph=g,
    dataloader=ds,  # can be Dataset or DataLoader
    metric=partial(logit_diff,loss=False, mean=True),
    method="EAP-IG-inputs",
    ig_steps=5,
    is_absa=True,
    batch_size=5,
    device="mps"
)

In [ ]:
n_edges = g.real_edge_mask.sum().item()  # total 171K edges for qwen2.5-0.5B
for topk in [100, 200, 500, 1000, 2000, 5000, 10000, 20000]:
    g.reset()
    g.apply_topn(topk, True)
    results = evaluate_graph_multitoken(model=model,
                                        graph=g,
                                        df=ds,  # your full dataset DataFrame
                                        metrics=[partial(logit_diff, mean=True, loss=False)],  # or just [logit_diff]
                                        batch_size=4,)

    print(f"with top-k = {topk} ({topk/n_edges:.1%}), the circuit's performance is {results}, faithfulness={results/baseline:.1%}")
    g.to_pt(f'outputs/opinion_circuit_topk-{topk}.pt')

    print(f"included nodes: {g.count_included_nodes()}, included edges: {g.count_included_edges()}")

# Circuit Merging

In [ ]:
from src.utils import edge_merging

graph_paths = ["outputs/multitokens/aspect_circuit_topk-5000.pt", "outputs/multitokens/sentiment_circuit_topk-5000.pt", "outputs/multitokens/opinion_circuit_topk-5000.pt"]

complete_edges = edge_merging(graph_paths=graph_paths)

complete_edges.to_csv("outputs/multitokens/complete_circuit_topk-5000.csv", index=None)

print(complete_edges.shape)

In [ ]:
import pandas as pd
from eap.graph import Graph

df = pd.read_csv("outputs/multitokens/complete_circuit_topk-2000.csv")
circuit_path = "outputs/multitokens/aspect_circuit_topk-2000.pt"
g = Graph.from_pt(circuit_path)
print(g.count_included_edges())
number_of_edge = g.count_included_edges()
for i, edge in enumerate(g.edges.values()):
    if edge.in_graph != True:
        if "m" in edge.child.name:
            mask = (df['parent_node'] == edge.parent.name) & (df['child_node'] == edge.child.name)
        else:
            mask = (df['parent_node'] == edge.parent.name) & (df['child_node'] == edge.child.name) & (df['child_type'] == edge.qkv)
        if sum(mask) == 1:
            edge.in_graph = True
            number_of_edge += 1
            print(i, number_of_edge, end="\r")
g.to_pt("outputs/multitokens/complete_circuit_topk-2000.pt")

In [ ]:
topk = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 30000, 40000]

for t in topk:
    print(t)
    df = pd.read_csv(f"outputs/multitokens/complete_circuit_topk-{t}.csv")

    a = []
    for row in df.iterrows():
        if "m" not in row[1]["child_node"] and "logits" not in row[1]["child_node"]:
            a.append(row[1]['child_node'] + ' ' + row[1]['child_type'])

    print(len(np.unique(a)))
    print(df.shape)
    print((df.shape[0]/179387)*100)
    print("\n")

# SFT

## SFT with Active Nodes

In [ ]:
import json
import torch
from torch.utils.data import Dataset
from transformer_lens.train import train
from transformer_lens.train import HookedTransformerTrainConfig
import pandas as pd
import gc
from src.utils import load_model, ABSAAutoRegressiveDataset, apply_active_edge

csv_path = "outputs/multitokens/complete_circuit_topk-2000.csv"
json_path = "hotel_dataset/hotel_aste_train_augmented_noreasoning.json"
model_name = "Qwen/Qwen2.5-0.5B"
max_len = 128
device = "mps"

# === Load ABSA Dataset ===
with open(json_path) as f:
    absa_data = json.load(f)

# === Load Model ===
model = load_model(model_name, device=device)

#we can try several number of samples
dataset = ABSAAutoRegressiveDataset(absa_data[:1000], model.tokenizer)

#comment it if you want to train full model
apply_active_edge(model, csv_path)


model.train()

# === Train Config ===
config = HookedTransformerTrainConfig(
    num_epochs=5,
    batch_size=32,
    lr=1e-5,
    device=device,
    print_every=10,
)

# === Start Training ===
trained_model = train(model, config, dataset)

### Inference

In [ ]:
model.generate([ "airnya kurang kencang . [A] [O] [S]"], 
               max_new_tokens=50,
               stop_at_eos=True,
               return_type="str")

### Simple Evaluation

In [ ]:
from src import filter_correct_data
import pandas as pd

# Check the accuracy using data existing counterfact data
# For the reference, the result from the finetune model using transformer is 48 out of 112 sample
dataset_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_corrected.csv"
test_df = pd.read_csv(dataset_path)

df_filtered = filter_correct_data(model, test_df, 
                                  "original_sentence", "original_triplet", filter_mode="AOS", filter_only_correct=False,)